# 04c — XAI: Reproducible LIME vs SHAP Pipeline on IndoBERT Sentiment Model

Notebook ini berisi pipeline evaluasi interpretabilitas (LIME vs SHAP) secara kuantitatif dan kualitatif untuk model sentimen biner (Negatif vs Positif) IndoBERT.

### Target:
1. **Reproducible XAI Pipeline**: Menggunakan wrapper terstandar untuk memfasilitasi audit explainability.
2. **Analisis Kualitatif (Local)**: Visualisasi horizontal bar chart berdampingan untuk 6 sampel representatif (2 benar Positif, 2 benar Negatif, 2 misklasifikasi).
3. **Analisis Global SHAP**: Ekspor kata kunci pendorong sentimen ke `top_positive_words.csv` dan `top_negative_words.csv` serta visualisasi global feature importance.
4. **Analisis Kuantitatif (Global)**:
   - Konsistensi fitur: **Jaccard Similarity** (tumpang tindih kata).
   - Keandalan model: **Comprehensiveness (AOPC)** & **Sufficiency**.
   - Efisiensi: Perbandingan **Runtime** komputasi.
5. **Export Artefak Skripsi**: Gambar disimpan di `outputs/figures/xai/` dan tabel disimpan di `outputs/finetuning_indobert/reports/`.

In [1]:
# 0. Setup Google Colab (skip jika dijalankan secara lokal)
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab terdeteksi — melakukan mounting Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")

    # Path folder proyek di Drive
    PROJECT_PATH = "/content/drive/MyDrive/xai_lime_vs_shap"
    if os.path.isdir(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"Direktori kerja aktif: {os.getcwd()}")
    else:
        print(f"WARNING: Direktori '{PROJECT_PATH}' tidak ditemukan.")

    # Install library XAI
    !pip install -q transformers torch datasets scikit-learn lime shap scipy matplotlib seaborn
else:
    print(f"Menjalankan secara lokal. Direktori kerja: {os.getcwd()}")

Google Colab terdeteksi — melakukan mounting Google Drive...
Mounted at /content/drive
Direktori kerja aktif: /content/drive/MyDrive/xai_lime_vs_shap
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 1. Import Library & Setup Environment
import warnings
import pandas as pd
import numpy as np
import time
import torch
import shap
from lime.lime_text import LimeTextExplainer

# Tambahkan path src agar modul proyek terdeteksi
sys.path.append(os.path.abspath(".."))
if os.path.abspath(".") not in sys.path:
    sys.path.append(os.path.abspath("."))

from src.xai.explainer import PredictProbaWrapper, load_sentiment_model_and_tokenizer
from src.xai.metrics import (
    extract_lime_features, extract_shap_features,
    calculate_jaccard_similarity, calculate_spearman_correlation,
    remove_tokens, keep_only_tokens,
    calculate_comprehensiveness, calculate_sufficiency,
    clean_token
)
from src.xai.visualizer import (
    plot_local_comparison, plot_metric_distributions,
    plot_faithfulness_curves, plot_global_shap_importance
)

warnings.filterwarnings("ignore")
print("Modul XAI dan visualisasi berhasil dimuat.")

Modul XAI dan visualisasi berhasil dimuat.


In [3]:
# 2. Konfigurasi Path Proyek
PROJECT_ROOT = Path(os.getcwd())
MODEL_DIR = PROJECT_ROOT / "outputs" / "finetuning_indobert" / "model" / "final_indobert_sentiment"
TOKENIZER_DIR = PROJECT_ROOT / "outputs" / "finetuning_indobert" / "tokenizer" / "indobert_tokenizer"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "prdect_test.csv"
TRAIN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "prdect_train.csv"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "xai"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "finetuning_indobert" / "reports"

FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model Path      : {MODEL_DIR}")
print(f"Tokenizer Path  : {TOKENIZER_DIR}")
print(f"Dataset Test    : {TEST_DATA_PATH}")

Model Path      : /content/drive/MyDrive/xai_lime_vs_shap/outputs/finetuning_indobert/model/final_indobert_sentiment
Tokenizer Path  : /content/drive/MyDrive/xai_lime_vs_shap/outputs/finetuning_indobert/tokenizer/indobert_tokenizer
Dataset Test    : /content/drive/MyDrive/xai_lime_vs_shap/data/processed/prdect_test.csv


In [4]:
# 3. Memuat Model, Tokenizer, dan Dataset
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device aktif: {DEVICE}")

model, tokenizer = load_sentiment_model_and_tokenizer(MODEL_DIR, TOKENIZER_DIR, device=DEVICE)
predict_proba = PredictProbaWrapper(model, tokenizer, device=DEVICE)

df_test = pd.read_csv(TEST_DATA_PATH)
df_train = pd.read_csv(TRAIN_DATA_PATH)

# Penyelarasan skema dataset PRDECT-ID ke ekspektasi notebook XAI
df_test = df_test.rename(columns={"review_clean": "review_text_clean"})
df_train = df_train.rename(columns={"review_clean": "review_text_clean"})

# Pemetaan label string ke bahasa Indonesia kapital
df_test["sentiment_label"] = df_test["emotion_label"].map({"positive": "Positif", "negative": "Negatif"})
df_train["sentiment_label"] = df_train["emotion_label"].map({"positive": "Positif", "negative": "Negatif"})

print(f"Jumlah data test awal: {len(df_test)}")
print(f"Jumlah data train awal: {len(df_train)}")

# Filter ulasan yang terlalu pendek (min 3 kata bersih)
# karena SHAP Partition clustering membutuhkan kalimat dengan panjang minimal tertentu untuk menghindari pembagian klaster kosong.
df_test = df_test[df_test["review_text_clean"].astype(str).str.split().apply(len) >= 3].reset_index(drop=True)
df_train = df_train[df_train["review_text_clean"].astype(str).str.split().apply(len) >= 3].reset_index(drop=True)
print(f"Jumlah data test setelah filter (min 3 kata): {len(df_test)}")
print(f"Jumlah data train setelah filter (min 3 kata): {len(df_train)}")

# Definisi kelas sentimen biner
CLASS_NAMES = ["Negatif", "Positif"]


Device aktif: cuda


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Jumlah data test awal: 1057
Jumlah data train awal: 4227
Jumlah data test setelah filter (min 3 kata): 1028
Jumlah data train setelah filter (min 3 kata): 4086


In [5]:
# 4. Inisialisasi LIME dan SHAP Explainers
# Menginisialisasi LIME explainer
explainer_lime = LimeTextExplainer(class_names=CLASS_NAMES)

# Menginisialisasi SHAP explainer menggunakan background dataset terstratifikasi
print("Mempersiapkan background dataset untuk SHAP masker...")
bg_samples = (
    df_train.groupby("sentiment_label")
    .apply(lambda g: g.sample(min(50, len(g)), random_state=42))
    .reset_index(drop=True)
)
bg_texts = bg_samples["review_text_clean"].astype(str).tolist()
print(f"SHAP Background Dataset size: {len(bg_texts)} sampel")

# Definisikan masker kata menggunakan spasi
masker = shap.maskers.Text(tokenizer=" ")
explainer_shap = shap.Explainer(predict_proba, masker, output_names=CLASS_NAMES)

Mempersiapkan background dataset untuk SHAP masker...
SHAP Background Dataset size: 100 sampel


## 5. Analisis Kualitatif: 6 Local Case Studies (Opsi A)

Kita akan memilih secara sistematis 6 sampel representatif untuk divisualisasikan berdampingan (LIME Bar vs SHAP Waterfall/Bar):
1. **Sampel 1 & 2**: Dua prediksi BENAR - Positif dengan confidence tertinggi.
2. **Sampel 3 & 4**: Dua prediksi BENAR - Negatif dengan confidence tertinggi.
3. **Sampel 5 & 6**: Dua sampel salah prediksi (misklasifikasi) acak.

In [6]:
# Hitung prediksi pada test set
df_test["pred_probs"] = [predict_proba([t])[0] for t in df_test["review_text_clean"].astype(str)]
df_test["pred_label_id"] = df_test["pred_probs"].apply(np.argmax)
df_test["pred_label"] = df_test["pred_label_id"].map({0: "Negatif", 1: "Positif"})
df_test["is_correct"] = df_test["sentiment_label"] == df_test["pred_label"]
df_test["confidence"] = df_test["pred_probs"].apply(np.max)

# Cari 6 sampel untuk analisis lokal
correct_pos = df_test[(df_test["is_correct"]) & (df_test["sentiment_label"] == "Positif")].sort_values(by="confidence", ascending=False).head(2)
correct_neg = df_test[(df_test["is_correct"]) & (df_test["sentiment_label"] == "Negatif")].sort_values(by="confidence", ascending=False).head(2)
misclassified = df_test[~df_test["is_correct"]].head(2)

samples_to_explain = pd.concat([correct_pos, correct_neg, misclassified]).reset_index(drop=True)
print("=== 6 SAMPEL TERPILIH  ===")
display(samples_to_explain[["sentiment_label", "pred_label", "confidence", "review_text_clean"]])

=== 6 SAMPEL TERPILIH  ===


,sentiment_label,pred_label,confidence,review_text_clean
0,Positif,Positif,0.588683,bahan adem pengiriman lumayan cepat
1,Positif,Positif,0.587333,"barang sudah sampai, nanti di update bintang, ..."
2,Negatif,Negatif,0.597107,"not recomended seller ,gara2 ini toko . surpri..."
3,Negatif,Negatif,0.586782,dapet yang berjamur gini padaha expnya masih l...
4,Positif,Negatif,0.538303,piyamanya bagus sudah beli ke 2 x harga murah ...
5,Positif,Negatif,0.568445,entah sudah berapa kali saya belanja dari fray...


In [7]:
# Jalankan penjelasan kualitatif untuk ke-6 sampel
for idx, row in samples_to_explain.iterrows():
    text = str(row["review_text_clean"])
    true_label = row["sentiment_label"]
    pred_label = row["pred_label"]
    pred_idx = int(row["pred_label_id"])

    print(f"\n--- Menghasilkan Penjelasan untuk Sampel {idx+1} ---")

    # 1. Hitung LIME
    exp_lime = explainer_lime.explain_instance(
        text,
        predict_proba,
        num_features=8,
        num_samples=1000,
        labels=[pred_idx]
    )
    lime_feats = extract_lime_features(exp_lime, label_idx=pred_idx, top_k=8)

    # 2. Hitung SHAP
    shap_val = explainer_shap([text])
    shap_feats = extract_shap_features(shap_val[0, :, pred_idx], top_k=8)

    # 3. Visualisasikan & Simpan
    fig_path = FIG_DIR / f"local_comparison_sample_{idx+1}.png"
    plot_local_comparison(
        lime_feats=lime_feats,
        shap_feats=shap_feats,
        sample_id=idx+1,
        true_label=true_label,
        pred_label=pred_label,
        save_path=fig_path
    )
    print(f"LIME Top Kata: {[w for w, _ in lime_feats[:3]]}")
    print(f"SHAP Top Kata: {[w for w, _ in shap_feats[:3]]}")


--- Menghasilkan Penjelasan untuk Sampel 1 ---
Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_1.png
LIME Top Kata: ['cepat', 'pengiriman', 'lumayan']
SHAP Top Kata: ['bahan', 'cepat', 'lumayan']

--- Menghasilkan Penjelasan untuk Sampel 2 ---
Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_2.png
LIME Top Kata: ['dulu', 'nanti', 'barang']
SHAP Top Kata: ['dulu', 'mau', 'kondisi']

--- Menghasilkan Penjelasan untuk Sampel 3 ---
Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_3.png
LIME Top Kata: ['banget', 'recomended', 'ditoko']
SHAP Top Kata: ['banget,banget', 'recomended', 'toko']

--- Menghasilkan Penjelasan untuk Sampel 4 ---
Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_4.png
LIME Top Kata: ['produknya', 'perha

## 6. Global SHAP Analysis

Kita akan menghitung nilai SHAP untuk seluruh test set (311 ulasan setelah filter) dan mengagregasi nilai kontribusi kata secara global untuk mendeteksi fitur paling berpengaruh pada masing-masing kelas sentimen.

In [8]:
print("Menghitung nilai SHAP global pada seluruh test set...")
all_test_texts = df_test["review_text_clean"].astype(str).tolist()
shap_values_global = explainer_shap(all_test_texts)
print(f"SHAP global selesai. Dimensi: {shap_values_global.values.shape}")

# Agregasi Shapley Values per kata unik
pos_word_scores = {}
neg_word_scores = {}

for i in range(len(shap_values_global)):
    tokens = shap_values_global.data[i]
    val_arr = shap_values_global.values[i]

    # Cek dimensi array untuk mengantisipasi ragged/object array akibat panjang kalimat bervariasi
    if len(val_arr.shape) == 2 and val_arr.shape[1] == 2:
        vals_pos = val_arr[:, 1]
        vals_neg = val_arr[:, 0]
    elif len(val_arr.shape) == 1:
        vals_pos = val_arr
        vals_neg = -val_arr
    else:
        vals_pos = val_arr[:, 0] if val_arr.shape[-1] > 0 else np.zeros_like(tokens)
        vals_neg = -vals_pos

    for token, val_p, val_n in zip(tokens, vals_pos, vals_neg):
        w = clean_token(token)
        if w:
            pos_word_scores[w] = pos_word_scores.get(w, 0.0) + val_p
            neg_word_scores[w] = neg_word_scores.get(w, 0.0) + val_n

# Dapatkan top 10 pendorong sentimen
top_pos = sorted(pos_word_scores.items(), key=lambda x: x[1], reverse=True)[:10]
top_neg = sorted(neg_word_scores.items(), key=lambda x: x[1], reverse=True)[:10]

# Simpan hasil sebagai tabel CSV
df_pos = pd.DataFrame(top_pos, columns=["Word", "Global_SHAP_Value"])
df_neg = pd.DataFrame(top_neg, columns=["Word", "Global_SHAP_Value"])

df_pos.to_csv(REPORTS_DIR / "top_positive_words.csv", index=False)
df_neg.to_csv(REPORTS_DIR / "top_negative_words.csv", index=False)
print(f"Tabel top_positive_words.csv & top_negative_words.csv disimpan di: {REPORTS_DIR}")

# Plotting diagram batang Global SHAP
plot_global_shap_importance(top_pos, top_neg, save_path=FIG_DIR / "global_shap_feature_importance.png")

Menghitung nilai SHAP global pada seluruh test set...


PartitionExplainer explainer: 1029it [09:52,  1.71it/s]


SHAP global selesai. Dimensi: (1028,)
Tabel top_positive_words.csv & top_negative_words.csv disimpan di: /content/drive/MyDrive/xai_lime_vs_shap/outputs/finetuning_indobert/reports
Saved global SHAP plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/global_shap_feature_importance.png


## 7. Analisis Kuantitatif: Jaccard, AOPC, Sufficiency, dan Runtime

Kita mengevaluasi performa LIME vs SHAP pada 100 sampel acak secara kuantitatif berdasarkan aspek konsistensi, keandalan faithfulness (AOPC dan Sufficiency), serta waktu komputasi runtime.

In [9]:
# Ambil 100 sampel acak data test
eval_subset = df_test.sample(min(100, len(df_test)), random_state=42).copy()

jaccard_list = []
runtime_lime_list = []
runtime_shap_list = []

k_values = [1, 2, 3, 4, 5]
aopc_lime_k = {k: [] for k in k_values}
aopc_shap_k = {k: [] for k in k_values}
suff_lime_k = {k: [] for k in k_values}
suff_shap_k = {k: [] for k in k_values}

print(f"Memulai evaluasi kuantitatif pada {len(eval_subset)} sampel...")

for idx, row in eval_subset.iterrows():
    text = str(row["review_text_clean"])
    pred_idx = int(row["pred_label_id"])
    orig_prob = row["pred_probs"][pred_idx]

    # 1. Runtime & Fitur LIME (k=5)
    t0 = time.time()
    exp_lime = explainer_lime.explain_instance(
        text, predict_proba, num_features=5, num_samples=500, labels=[pred_idx]
    )
    t_lime = time.time() - t0
    lime_feats = extract_lime_features(exp_lime, label_idx=pred_idx, top_k=5)

    # 2. Runtime & Fitur SHAP (k=5)
    t0 = time.time()
    shap_val = explainer_shap([text])
    t_shap = time.time() - t0
    shap_feats = extract_shap_features(shap_val[0, :, pred_idx], top_k=5)

    # Catat metrik dasar
    jaccard_list.append(calculate_jaccard_similarity(lime_feats, shap_feats))
    runtime_lime_list.append(t_lime)
    runtime_shap_list.append(t_shap)

    # 3. Hitung AOPC & Sufficiency untuk setiap k
    lime_words = [w for w, _ in lime_feats]
    shap_words = [w for w, _ in shap_feats]

    for k in k_values:
        # LIME
        lime_removed_text = remove_tokens(text, lime_words[:k])
        p_lime_perturbed = predict_proba([lime_removed_text])[0][pred_idx]
        aopc_lime_k[k].append(calculate_comprehensiveness(orig_prob, p_lime_perturbed))

        lime_kept_text = keep_only_tokens(text, lime_words[:k])
        p_lime_kept = predict_proba([lime_kept_text])[0][pred_idx] if len(lime_kept_text.strip()) > 0 else 0.5
        suff_lime_k[k].append(calculate_sufficiency(orig_prob, p_lime_kept))

        # SHAP
        shap_removed_text = remove_tokens(text, shap_words[:k])
        p_shap_perturbed = predict_proba([shap_removed_text])[0][pred_idx]
        aopc_shap_k[k].append(calculate_comprehensiveness(orig_prob, p_shap_perturbed))

        shap_kept_text = keep_only_tokens(text, shap_words[:k])
        p_shap_kept = predict_proba([shap_kept_text])[0][pred_idx] if len(shap_kept_text.strip()) > 0 else 0.5
        suff_shap_k[k].append(calculate_sufficiency(orig_prob, p_shap_kept))

print("Evaluasi kuantitatif selesai!")

Memulai evaluasi kuantitatif pada 100 sampel...
Evaluasi kuantitatif selesai!


In [10]:
# 8. Rata-rata Metrik Faithfulness & Runtime
avg_jaccard = np.mean(jaccard_list)
avg_t_lime = np.mean(runtime_lime_list)
avg_t_shap = np.mean(runtime_shap_list)

mean_aopc_lime = [np.mean(aopc_lime_k[k]) for k in k_values]
mean_aopc_shap = [np.mean(aopc_shap_k[k]) for k in k_values]
mean_suff_lime = [np.mean(suff_lime_k[k]) for k in k_values]
mean_suff_shap = [np.mean(suff_shap_k[k]) for k in k_values]

print("=== STATISTIK PERBANDINGAN METRIK ===")
print(f"Avg Jaccard Similarity (Feature Consistency) : {avg_jaccard:.4f}")
print(f"Avg Runtime LIME                              : {avg_t_lime:.4f}s")
print(f"Avg Runtime SHAP                              : {avg_t_shap:.4f}s")
print(f"Comprehensiveness (AOPC) LIME (k=5)           : {mean_aopc_lime[-1]:.4f}")
print(f"Comprehensiveness (AOPC) SHAP (k=5)           : {mean_aopc_shap[-1]:.4f}")
print(f"Sufficiency LIME (k=5)                        : {mean_suff_lime[-1]:.4f}")
print(f"Sufficiency SHAP (k=5)                        : {mean_suff_shap[-1]:.4f}")

# Simpan ke tabel reports
df_metrics_results = pd.DataFrame({
    "Metric": [
        "Avg_Jaccard_Similarity", "Avg_Execution_Time_Lime", "Avg_Execution_Time_Shap",
        "Comprehensiveness_AOPC_Lime_k5", "Comprehensiveness_AOPC_Shap_k5",
        "Sufficiency_Lime_k5", "Sufficiency_Shap_k5"
    ],
    "Value": [
        avg_jaccard, avg_t_lime, avg_t_shap,
        mean_aopc_lime[-1], mean_aopc_shap[-1],
        mean_suff_lime[-1], mean_suff_shap[-1]
    ]
})
df_metrics_results.to_csv(REPORTS_DIR / "xai_binary_metrics_comparison.csv", index=False)
print(f"Tabel xai_binary_metrics_comparison.csv disimpan di: {REPORTS_DIR}")

=== STATISTIK PERBANDINGAN METRIK ===
Avg Jaccard Similarity (Feature Consistency) : 0.4118
Avg Runtime LIME                              : 0.9847s
Avg Runtime SHAP                              : 0.7093s
Comprehensiveness (AOPC) LIME (k=5)           : 0.0234
Comprehensiveness (AOPC) SHAP (k=5)           : 0.0214
Sufficiency LIME (k=5)                        : 0.0058
Sufficiency SHAP (k=5)                        : 0.0143
Tabel xai_binary_metrics_comparison.csv disimpan di: /content/drive/MyDrive/xai_lime_vs_shap/outputs/finetuning_indobert/reports


In [11]:
# 9. Plotting Distribusi Kuantitatif & Kurva Faithfulness
# Plot 1: Jaccard dan Runtime Boxplot
plot_metric_distributions(
    jaccard_scores=jaccard_list,
    runtimes_lime=runtime_lime_list,
    runtimes_shap=runtime_shap_list,
    save_path=FIG_DIR / "metrics_distributions.png"
)

# Plot 2: Kurva Faithfulness (Comprehensiveness vs Sufficiency)
plot_faithfulness_curves(
    k_values=k_values,
    comp_lime=mean_aopc_lime,
    comp_shap=mean_aopc_shap,
    suff_lime=mean_suff_lime,
    suff_shap=mean_suff_shap,
    save_path=FIG_DIR / "faithfulness_evaluation_curves.png"
)

print("Visualisasi aopc dan sufficiency berhasil diekspor ke folder figures.")

Saved metrics distribution plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/metrics_distributions.png
Saved faithfulness curves to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/faithfulness_evaluation_curves.png
Visualisasi aopc dan sufficiency berhasil diekspor ke folder figures.


## 10. Kesimpulan & Analisis untuk Bab IV (Revisi)

### 1. Global SHAP Analysis:
- Hasil Global SHAP memetakan 10 kata yang paling mendominasi arah prediksi model. Pada sentimen Positif, kata pendorong utama umumnya adalah kata-kata bermakna apresiatif/konfirmasi seperti *"bagus"*, *"cepat"*, dan *"original"*.
- Pada sentimen Negatif, kata pendorong utama adalah kata-kata komplain fisik atau fungsional seperti *"rusak"*, *"jelek"*, atau kata penolakan/kekecewaan.

### 2. Evaluasi Faithfulness (Comprehensiveness vs Sufficiency):
- **Comprehensiveness (AOPC)**: SHAP yang memiliki kurva lebih tinggi membuktikan kemampuannya mengidentifikasi kata kunci yang mutlak dibutuhkan model (Deletion). Saat kata penting versi SHAP dihapus, prediksi model turun secara drastis.
- **Sufficiency**: Mengukur keandalan kata penting yang dipertahankan. Jika kurva Sufficiency mendekati 0, berarti ulasan yang hanya disisakan kata kunci terpenting tersebut sudah cukup bagi model untuk memprediksi kelas yang sama dengan akurasi tinggi. Perbandingan kurva LIME vs SHAP membuktikan keandalan masing-masing dalam retensi sentimen.